In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 单个专家网络
class Expert(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(Expert, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = self.layer2(x)
        return x

# 门控网络
class Gate(nn.Module):
    def __init__(self, input_dim, num_experts):
        super(Gate, self).__init__()
        self.layer = nn.Linear(input_dim, num_experts)
        
    def forward(self, x):
        return F.softmax(self.layer(x), dim=-1)

# MoE 模型
class MixtureOfExperts(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_experts):
        super(MixtureOfExperts, self).__init__()
        self.num_experts = num_experts
        
        # 创建多个专家网络
        self.experts = nn.ModuleList([
            Expert(input_dim, hidden_dim, output_dim) 
            for _ in range(num_experts)
        ])
        
        # 创建门控网络
        self.gate = Gate(input_dim, num_experts)
        
    def forward(self, x):
        # 获取门控权重 [batch_size, num_experts]
        gate_weights = self.gate(x)
        
        # 获取每个专家的输出 [batch_size, num_experts, output_dim]
        expert_outputs = torch.stack(
            [expert(x) for expert in self.experts], 
            dim=1
        )
        
        # 计算加权输出 [batch_size, output_dim]
        output = torch.einsum('be,b eo->bo', gate_weights, expert_outputs)
        return output

# 训练示例
def train_moe():
    # 超参数
    input_dim = 10
    hidden_dim = 20
    output_dim = 5
    num_experts = 3
    batch_size = 32
    num_epochs = 100
    
    # 创建模型
    model = MixtureOfExperts(input_dim, hidden_dim, output_dim, num_experts)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    # 生成示例数据
    X = torch.randn(1000, input_dim)
    y = torch.randn(1000, output_dim)
    
    # 训练循环
    for epoch in range(num_epochs):
        # 生成随机批次
        idx = torch.randperm(X.size(0))[:batch_size]
        batch_x = X[idx]
        batch_y = y[idx]
        
        # 前向传播
        output = model(batch_x)
        loss = criterion(output, batch_y)
        
        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# 测试代码
if __name__ == "__main__":
    # 设置随机种子以确保可重复性
    torch.manual_seed(42)
    
    # 运行训练
    train_moe()
    
    # 测试单个输入
    model = MixtureOfExperts(10, 20, 5, 3)
    test_input = torch.randn(1, 10)
    output = model(test_input)
    print("\n测试输出:")
    print(output)

Epoch [10/100], Loss: 1.2098
Epoch [20/100], Loss: 0.9936
Epoch [30/100], Loss: 0.9184
Epoch [40/100], Loss: 1.0288
Epoch [50/100], Loss: 1.0312
Epoch [60/100], Loss: 1.1057
Epoch [70/100], Loss: 1.0355
Epoch [80/100], Loss: 1.0266
Epoch [90/100], Loss: 1.0111
Epoch [100/100], Loss: 1.1912

测试输出:
tensor([[ 0.0722,  0.0394, -0.2069,  0.1519,  0.0538]],
       grad_fn=<ViewBackward0>)
